# Garment verification v2 — full catalog
Enable Internet and GPU T4 x2. Attach the v2 ZIP. This processes unique URLs and preserves original IDs. No more mandatory pilot rounds. Relative prompt scores are not accuracy guarantees.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, shutil, json, hashlib
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
unpack=Path('/kaggle/working/garment_code_v2')
candidates=[p for p in Path('/kaggle/input').rglob('garment_core.py') if "VERSION='2.0.0'" in p.read_text()]
if not candidates:
    archives=list(Path('/kaggle/input').rglob('shirt_garment_verification_v2.zip'))
    assert len(archives)==1, 'Attach the v2 ZIP.'
    unpack.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        assert all((unpack/n).resolve().is_relative_to(unpack.resolve()) for n in z.namelist())
        z.extractall(unpack)
    candidates=list(unpack.rglob('garment_core.py'))
assert len(candidates)==1, f'Select one v2 code package: {candidates}'
CODE=candidates[0].parent
DATA=CODE/'input/shirts_master.parquet'
SCRIPT=CODE/'classify_garments.py'
OUT=Path('/kaggle/working/garment_results_v2')
manifest=json.loads((CODE/'package_manifest.json').read_text())
assert hashlib.sha256(DATA.read_bytes()).hexdigest()==manifest['master_sha256']
RESTORE_ZIP=None # Optional Path('/kaggle/input/.../garment_results_v2_backup.zip')
if RESTORE_ZIP:
    assert not (OUT/'checkpoint.sqlite').exists(), 'Existing checkpoint: do not overwrite it.'
    with zipfile.ZipFile(RESTORE_ZIP) as z:
        base=OUT.parent.resolve()
        assert all((base/n).resolve().is_relative_to(base) for n in z.namelist())
        assert all(n.startswith(OUT.name+'/') for n in z.namelist())
        z.extractall(base)
DEPS=Path('/kaggle/working/garment_deps_v2')
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--upgrade','--target',str(DEPS),'-r',str(CODE/'requirements-kaggle.txt')],check=True)
ENV=os.environ.copy()
ENV['PYTHONPATH']=str(DEPS)+os.pathsep+ENV.get('PYTHONPATH','')
print('Code:',CODE,'Data:',DATA,'Results:',OUT,sep='\n')


In [ ]:
import torch
assert torch.cuda.is_available() and torch.cuda.device_count()>=2, 'Select GPU T4 x2.'
print(torch.__version__,[torch.cuda.get_device_name(i) for i in range(2)])
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(CODE/'tests'),'-v'],env=ENV,check=True)


## Full catalog
Run this cell. LIMIT=0 processes every remaining unique URL. LIMIT=12 is available for a short optional speed check; change it back to 0 to finish. Existing v2 checkpoint results resume automatically.


In [ ]:
LIMIT=0
subprocess.run([sys.executable,'-u',str(SCRIPT),'run','--input',str(DATA),
    '--output-dir',str(OUT),'--devices','0,1','--batch-size','24',
    '--download-workers','12','--min-score','0.85','--min-margin','0.20',
    '--limit-images',str(LIMIT),'--preview-count','100'],env=ENV,check=True)
print((OUT/'summary.json').read_text())


## Download results
After the worker exits, run this cell and download the linked ZIP. Keep uncertain products separate; do not replace the master blindly.


In [ ]:
from IPython.display import FileLink, display
assert OUT.exists()
archive=shutil.make_archive('/kaggle/working/garment_results_v2_backup','zip',OUT.parent,OUT.name)
display(FileLink(archive))
